<a href="https://colab.research.google.com/github/n-zad/av-vlm-belief-persistence/blob/feat%2Feval-pipeline/eval-pipeline/notebooks/colab_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Belief-persistence evaluation on Colab

**Before running:**
1. *Runtime → Change runtime type → T4 GPU*.
2. Upload `data/cam_front_images.zip` (the 804 front-camera images, 118 MB) to Google Drive at `MyDrive/cmpe249/`.

Results are written to that Drive folder, and `run` skips finished sequences, so if Colab disconnects
just reconnect and run the cells again; it continues where it stopped.

In [4]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Settings

In [5]:
DRIVE_DIR = "/content/drive/MyDrive/cmpe249"   # holds cam_front_images.zip; results go here too
BRANCH = "feat/eval-pipeline"

# Run once per model: Qwen first, then switch to the LLaVA lines and run from step 3 again.
MODEL, TAG = "Qwen/Qwen2.5-VL-7B-Instruct", "qwen7b"
# MODEL, TAG = "llava-hf/llava-onevision-qwen2-7b-ov-hf", "llava7b"

RESULTS = f"{DRIVE_DIR}/results_{TAG}.jsonl"

## 2. Setup: Drive, code, packages, images

In [6]:
from google.colab import drive
drive.mount("/content/drive")

!if [ -d /content/repo ]; then git -C /content/repo pull -q; else git clone -q -b {BRANCH} https://github.com/n-zad/av-vlm-belief-persistence.git /content/repo; fi
%cd /content/repo
!pip install -q -U transformers accelerate bitsandbytes
!mkdir -p data/nuscenes && unzip -q -o {DRIVE_DIR}/cam_front_images.zip -d data/nuscenes
!echo "images: $(ls data/nuscenes/samples/CAM_FRONT | wc -l) (expect 804)"

Mounted at /content/drive
/content/repo
images: 804 (expect 804)


## 3. Smoke test: one sequence, all three modes

Check that the raw answers are sensible JSON before the long run. If they are empty or gibberish,
stop here and report it. If you get a CUDA out-of-memory error, add `--max-side 768` to both run commands.

In [7]:
!rm -f /content/smoke.jsonl
!python -m belief_eval run --dataroot data/nuscenes --sequences sequences_trainval.jsonl --model {MODEL} --4bit --limit 1 --out /content/smoke.jsonl

import json
for rec in map(json.loads, open("/content/smoke.jsonl")):
    print(f"--- {rec['mode']} (evidence frame {rec['evidence_index']})")
    for f in rec["frames"]:
        print(f"t={f['t']} present={f['present']} action={f['action']} gt_action={f['gt_action']} | {f['raw'][:150]!r}")

config.json: 100% 1.37k/1.37k [00:00<00:00, 4.22MB/s]
model.safetensors.index.json: 100% 57.6k/57.6k [00:00<00:00, 117MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/7.73G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/16.6G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.6G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.6G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   4% 715M/16.6G [00:10<04:07, 64.2MB/s,  134MB/s  ]
Reconstructing (incomplete total...):   7% 1.10G/16.6G [00:14<03:16, 78.7MB/s,  103MB/s  ]
Reconstructing (incomplete total...):   8% 1.41G/16.6G [00:18<04:11, 60.4MB/s, 97.6MB/s  ]

Reconstructing (incomplete total...):  39% 6.47G/16.6G [01:01<04:14, 39.8MB/s, 44.6MB/s  ]
Reconstructing (incomplete total...):  44% 7.22G/16.6G [01:09<02:50, 54.7MB/s, 67.1MB/s  ]
Reconstr

## 4. Full run: 145 sequences × 3 modes

Expect several hours per model. Safe to re-run after a disconnect.

In [8]:
!python -m belief_eval run --dataroot data/nuscenes --sequences sequences_trainval.jsonl --model {MODEL} --4bit --out {RESULTS}
!echo "finished records: $(wc -l < {RESULTS}) / 435"

Loading weights: 100% 729/729 [01:11<00:00, 10.27it/s]
results -> /content/drive/MyDrive/cmpe249/results_qwen7b.jsonl
finished records: 435 / 435


## 5. Scores (all models run so far)

In [9]:
!python -m belief_eval score {DRIVE_DIR}/results_*.jsonl

{
  "Qwen/Qwen2.5-VL-7B-Instruct": {
    "reconsider": {
      "n_sequences": 145,
      "n_hallucinated": 139,
      "persistence_rate": 0.9640287769784173,
      "persistence_rate_ci": [
        0.9280575539568345,
        0.9928057553956835
      ],
      "latency_median": 2,
      "latency_p90": 4,
      "n_never_corrected": 126,
      "post_evidence_accuracy": 0.06712643678160919,
      "post_evidence_accuracy_ci": [
        0.0367816091954023,
        0.10367816091954023
      ],
      "frame_accuracy": 0.05747126436781609,
      "safety_balanced_accuracy": 0.5335325538504367,
      "unsafe_action_rate": 0.034482758620689655
    },
    "stateful": {
      "n_sequences": 145,
      "n_hallucinated": 139,
      "persistence_rate": 0.9568345323741008,
      "persistence_rate_ci": [
        0.920863309352518,
        0.9856115107913669
      ],
      "latency_median": 2,
      "latency_p90": 4,
      "n_never_corrected": 122,
      "post_evidence_accuracy": 0.09195402298850575,
     